In [ ]:
#Setup for Target Data:
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Load PM2.5 target dataset
pm25_path = "/content/drive/MyDrive/ResearchPG/Datasets/Daily_Data.csv"
pm25 = pd.read_csv(pm25_path)

# Check columns
print("Columns:", pm25.columns.tolist())

# Convert date column
pm25["Date"] = pd.to_datetime(pm25["Date"])

# Select Santa Cruz site
target_site_id = 60870007
target_site = pm25[pm25["Site ID"] == target_site_id].copy()

# Get latitude and longitude from the correct columns
site_lat = target_site["Site Latitude"].iloc[0]
site_lon = target_site["Site Longitude"].iloc[0]

print("Target Site ID:", target_site_id)
print("Latitude:", site_lat)
print("Longitude:", site_lon)

# Keep only needed columns
target_site = target_site[
    ["Date", "Site ID", "Site Latitude", "Site Longitude", "Daily Mean PM2.5 Concentration"]
].copy()

target_site = target_site.rename(columns={
    "Site Latitude": "Latitude",
    "Site Longitude": "Longitude",
    "Daily Mean PM2.5 Concentration": "pm25"
})

print(target_site.head())
print("Total rows:", len(target_site))

In [ ]:
#Authenticate the Earth Engine API:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='earth-engine-api-492523') #Add your project ID in the quotes.

In [ ]:
#Create a PM2.5 vegetation region around the site and load MODIS NDVI (Google Earth Engine's Vegetation Data)

#Create a point from the PM2.5 site coordinates:
site_point = ee.Geometry.Point([site_lon, site_lat])

#Create a region around the site
#50000 meters = 50 km buffer
region = site_point.buffer(50000).bounds()

#Load MODIS NDVI for 2025 (or whatever your dates are for your dataset:)
modis = (
    ee.ImageCollection("MODIS/061/MOD13Q1")
    .filterDate("2025-01-01", "2026-01-01")
    .select("NDVI")
)

In [ ]:
#Export the vegetation data as a CSV time series:
def image_to_feature(img):
    stats = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=250,
        maxPixels=1e13
    )
    return ee.Feature(None, {
        "date": ee.Date(img.get("system:time_start")).format("YYYY-MM-dd"),
        "NDVI": stats.get("NDVI")
    })

ndvi_fc = modis.map(image_to_feature)

task = ee.batch.Export.table.toDrive(
    collection=ndvi_fc,
    description="NDVI_timeseries",
    folder="EarthEngineExports",
    fileFormat="CSV"
)

task.start()
print("Export started!")

In [ ]:
#Check export status:
print(task.status())

In [ ]:
#Load the exported NDVI CSV from Drive:
import pandas as pd

#Change this path if needed:
ndvi_path = "/content/drive/MyDrive/EarthEngineExports/NDVI_timeseries.csv"

ndvi = pd.read_csv(ndvi_path)
print(ndvi.head())
print(ndvi.columns)
print("Original rows:", len(ndvi))

In [ ]:
print(ndvi.columns)
print(ndvi.head())

In [ ]:
# Convert all the NDVI data to daily data for 2025

# Convert the date column
ndvi["date"] = pd.to_datetime(ndvi["date"])

# Keep only the columns you actually need
ndvi = ndvi[["date", "NDVI"]].copy()

# Convert NDVI to numeric
ndvi["NDVI"] = pd.to_numeric(ndvi["NDVI"], errors="coerce")

# Set as index
ndvi = ndvi.set_index("date")

# Get the full daily date range for 2025
full_dates = pd.date_range(start="2025-01-01", end="2025-12-31")
ndvi = ndvi.reindex(full_dates)

# Rename the index
ndvi.index.name = "date"

# Insert missing values
ndvi_daily = ndvi.interpolate(method="linear")

# Reset the index
ndvi_daily = ndvi_daily.reset_index()

print(ndvi_daily.head())
print(ndvi_daily.tail())
print("Total rows:", len(ndvi_daily))
print(ndvi_daily.dtypes)

In [ ]:
#Save the daily vegetation data:
ndvi_daily_path = "/content/drive/MyDrive/ResearchPG/Datasets/NDVI_daily.csv"
ndvi_daily.to_csv(ndvi_daily_path, index=False)

print("Saved daily NDVI to:", ndvi_daily_path)

In [ ]:
#Merge the target PM2.5 data with the daily vegetation data:

#When you download your data, make sure your dates match!
target_site["Date"] = pd.to_datetime(target_site["Date"])
ndvi_daily["date"] = pd.to_datetime(ndvi_daily["date"])

#Merge the two datasets:
merged_pm25_ndvi = target_site.merge(
    ndvi_daily,
    left_on = "Date",
    right_on = "date",
    how = "left"

)

#Drop duplicate date column (this is optional, but for the sake of this project, I will do it:)
merged_pm25_ndvi = merged_pm25_ndvi.drop(columns=["date"])

print(merged_pm25_ndvi.head())
print("Merged rows:", len(merged_pm25_ndvi))

In [ ]:
#Save the merged PM2.5 and the vegetation dataset:

merged_path = "/content/drive/MyDrive/ResearchPG/Datasets/PM25_NDVI_merged.csv"
merged_pm25_ndvi.to_csv(merged_path, index=False)

print("Saved merged data to:", merged_path)

In [ ]:
#Pull in hourly data for HRRR, which is our weather variables. The first step is to install Herbie and weather dependencies for HRRR
!pip install -q herbie-data==2024.3.0 pandas==2.2.2 cfgrib xarray netcdf4

In [ ]:
#Check the version of pandas (just in case!)

import pandas as pd
print(pd.__version__)

In [ ]:
#Load both Wildfire CSVs

import pandas as pd

fire_j1_path = "/content/drive/MyDrive/ResearchPG/Datasets/WildfireData_J1 VIIRS C2 - fire_archive_J1V-C2_729304.csv"
fire_suomi_path = "/content/drive/MyDrive/ResearchPG/Datasets/Wildfire_Data_SUOMI VIIRS C2 - fire_archive_SV-C2_729305.csv"

fire_j1 = pd.read_csv(fire_j1_path)
fire_suomi = pd.read_csv(fire_suomi_path)

print("J1 columns:", fire_j1.columns.tolist())
print("SUOMI columns:", fire_suomi.columns.tolist())
print("J1 rows:", len(fire_j1))
print("SUOMI rows:", len(fire_suomi))

In [ ]:
#Combine the two datasets

fire_j1["source_name"] = "J1"
fire_suomi["source_name"] = "SUOMI"

fire_all = pd.concat([fire_j1, fire_suomi], ignore_index=True)

print(fire_all.head())
print("Total wildfire rows:", len(fire_all))

In [ ]:
#Clean the wildfire date and time columns
fire_all["acq_date"] = pd.to_datetime(fire_all["acq_date"])
fire_all["acq_time"] = fire_all["acq_time"].astype(str).str.zfill(4)

fire_all["datetime"] = pd.to_datetime(
    fire_all["acq_date"].dt.strftime("%Y-%m-%d") + " " +
    fire_all["acq_time"].str[:2] + ":" + fire_all["acq_time"].str[2:],
    errors="coerce"
)

print(fire_all[["acq_date", "acq_time", "datetime"]].head())

In [ ]:
#Filter all Wildfire data near the site:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

fire_all["distance_km"] = haversine(
    site_lat,
    site_lon,
    fire_all["latitude"],
    fire_all["longitude"]
)

fire_nearby = fire_all[fire_all["distance_km"] <= 100].copy()

print(fire_nearby.head())
print("Nearby wildfire rows:", len(fire_nearby))

In [ ]:
#Aggregate it to daily features
fire_daily = fire_nearby.groupby("acq_date").agg(
    fire_count=("acq_date", "size"),
    mean_frp=("frp", "mean"),
    max_frp=("frp", "max")
).reset_index()

fire_daily = fire_daily.rename(columns={"acq_date": "Date"})

print(fire_daily.head())
print("Daily wildfire rows:", len(fire_daily))

In [ ]:
#Force all Wildfire data to 2025
full_dates = pd.date_range(start="2025-01-01", end="2025-12-31")

fire_daily = fire_daily.set_index("Date").reindex(full_dates)
fire_daily.index.name = "Date"

fire_daily["fire_count"] = fire_daily["fire_count"].fillna(0)
fire_daily["mean_frp"] = fire_daily["mean_frp"].fillna(0)
fire_daily["max_frp"] = fire_daily["max_frp"].fillna(0)

fire_daily = fire_daily.reset_index()

print(fire_daily.head())
print(fire_daily.tail())
print("Total wildfire daily rows:", len(fire_daily))

In [ ]:
#Save the Wildfire Daily Dataset
fire_daily_path = "/content/drive/MyDrive/ResearchPG/Datasets/Wildfire_daily.csv"
fire_daily.to_csv(fire_daily_path, index=False)

print("Saved wildfire daily data to:", fire_daily_path)

In [ ]:
#Merge it into the PM2.5 + NDVI dataset:
merged_pm25_ndvi["Date"] = pd.to_datetime(merged_pm25_ndvi["Date"])
fire_daily["Date"] = pd.to_datetime(fire_daily["Date"])

merged_pm25_ndvi_fire = merged_pm25_ndvi.merge(fire_daily, on="Date", how="left")

print(merged_pm25_ndvi_fire.head())
print("Merged rows:", len(merged_pm25_ndvi_fire))

In [ ]:
#Save PM2.5 + NDVI + Wildfire merged dataset:
merged_fire_path = "/content/drive/MyDrive/ResearchPG/Datasets/PM25_NDVI_Wildfire_merged.csv"
merged_pm25_ndvi_fire.to_csv(merged_fire_path, index=False)

print("Saved merged dataset to:", merged_fire_path)

In [ ]:
#HRRR Setup
from herbie import Herbie
import pandas as pd
import numpy as np
import xarray as xr
from tqdm import tqdm
import os

#Make sure all of these dates are in datetime!
merged_pm25_ndvi_fire["Date"] = pd.to_datetime(merged_pm25_ndvi_fire["Date"])

#These will come from your first cell!
print("Using site latitude:", site_lat)
print("Using site longitude", site_lon)
print("Date range:", merged_pm25_ndvi_fire["Date"].min(), "to", merged_pm25_ndvi_fire["Date"].max())

In [ ]:
# Function to download HRRR weather variables for one day

def get_hrrr_for_date(date, site_lat, site_lon):
    """
    Pulls HRRR surface weather data for one date near the PM2.5 site.
    Uses the 12z run because it gives a consistent daily snapshot.
    """

    try:
        # HRRR analysis run for that date at 12:00 UTC
        h = Herbie(
            pd.to_datetime(date).strftime("%Y-%m-%d 12:00"),
            model="hrrr",
            product="sfc",
            fxx=0
        )

        # Pull common surface weather variables
        ds = h.xarray(
            ":(TMP|DPT):2 m|:(UGRD|VGRD):10 m|:GUST:surface|:PRES:surface"
        )

        # Select nearest grid point to your PM2.5 site
        point = ds.herbie.nearest_points(points=[(site_lon, site_lat)])

        row = {
            "Date": pd.to_datetime(date),
        }

        # Extract variables safely
        for var in point.data_vars:
            value = point[var].values

            if np.size(value) > 0:
                row[var] = float(np.ravel(value)[0])

        return row

    except Exception as e:
        print(f"Could not get HRRR for {date}: {e}")
        return {
            "Date": pd.to_datetime(date),
            "hrrr_error": str(e)
        }

In [ ]:
#HRRR data every 5th day because every day will overload the Colab
#Then, later, we can interpolate this data back to daily data.

merged_pm25_ndvi_fire["Date"] = pd.to_datetime(merged_pm25_ndvi_fire["Date"])

all_dates = sorted(merged_pm25_ndvi_fire["Date"].dropna().unique())

#You can change this number. I am using 5, but you can change it to 3 or 4 for more accuracy or to 7 if Colab crashes.
sample_step = 5

dates = all_dates[::sample_step]

print("Original daily data:", len(all_dates))
print("HRRR sampled data:", len(dates))
print("First sampled date:", dates[0])
print("Last sampled date:", dates[-1])

In [ ]:
#Lightweight HRRR pull function
# This avoids using xarray, which was causing memory crashes.
# It only downloads a very small subset of HRRR data (2m temperature).

import os
import gc
import pandas as pd
import numpy as np
from herbie import Herbie

def get_hrrr_minimal(date):
    """
    Downloads a very small HRRR subset for a single date.
    Does not load the full dataset into memory.
    """

    row = {"Date": pd.to_datetime(date)}

    try:
        h = Herbie(
            pd.to_datetime(date).strftime("%Y-%m-%d 12:00"),
            model="hrrr",
            product="sfc",
            fxx=0
        )

        # Download only temperature at 2 meters above ground
        grib_file = h.download(searchString=":TMP:2 m above ground:")

        # At this stage, we are only confirming the file downloads successfully
        # We are not extracting values yet to avoid memory issues
        row["temp_2m_K"] = np.nan

        # Immediately delete file to prevent disk/memory buildup
        if os.path.exists(grib_file):
            os.remove(grib_file)

        gc.collect()

    except Exception as e:
        print("Failed:", date, e)
        row["temp_2m_K"] = np.nan

    return row

In [ ]:
# Controlled loop to test HRRR downloads safely
# This runs on only a few dates to confirm stability before scaling up.

from tqdm import tqdm

hrrr_rows = []

# Use only a few dates to prevent crashing during testing
test_dates = dates[:3]

for date in tqdm(test_dates):
    row = get_hrrr_minimal(date)
    hrrr_rows.append(row)

    # Save progress after each iteration
    pd.DataFrame(hrrr_rows).to_csv(
        "/content/drive/MyDrive/ResearchPG/Datasets/HRRR_partial_sampled.csv",
        index=False
    )

    print("Saved:", date)

    gc.collect()

hrrr_sampled = pd.DataFrame(hrrr_rows)

print("Completed test run")
print(hrrr_sampled)

In [ ]:
# Function to extract one site value from a small HRRR GRIB file
# This uses xarray only after downloading one small HRRR variable subset.
# This is much safer than loading multiple HRRR variables at once.

import xarray as xr
import numpy as np
import gc
import os

def extract_point_from_grib(grib_file, site_lat, site_lon):
    """
    Opens a small HRRR GRIB file and extracts the nearest grid value
    to the PM2.5 monitoring site.
    """

    ds = None

    try:
        ds = xr.open_dataset(
            grib_file,
            engine="cfgrib",
            backend_kwargs={"indexpath": ""}
        )

        data_vars = list(ds.data_vars)

        if len(data_vars) == 0:
            return np.nan

        var_name = data_vars[0]

        lats = ds["latitude"].values
        lons = ds["longitude"].values

        target_lon = site_lon

        # HRRR longitude may use 0 to 360 instead of -180 to 180
        if np.nanmax(lons) > 180 and target_lon < 0:
            target_lon = target_lon + 360

        distance = (lats - site_lat) ** 2 + (lons - target_lon) ** 2
        y_index, x_index = np.unravel_index(np.nanargmin(distance), distance.shape)

        value = ds[var_name].values[y_index, x_index]

        return float(value)

    except Exception as e:
        print("Extraction failed:", e)
        return np.nan

    finally:
        if ds is not None:
            ds.close()

        gc.collect()

In [ ]:
#Function to extract multiple HRRR weather features for one date
# Each variable is downloaded separately to reduce memory pressure.

hrrr_variable_map = {
    "temp_2m_K": ":TMP:2 m above ground:",
    "dewpoint_2m_K": ":DPT:2 m above ground:",
    "u_wind_10m": ":UGRD:10 m above ground:",
    "v_wind_10m": ":VGRD:10 m above ground:"
}

def get_hrrr_features_for_date(date, site_lat, site_lon):
    """
    Downloads and extracts HRRR weather variables for one date.
    Saves only the nearest point value for the site.
    """

    row = {"Date": pd.to_datetime(date)}

    try:
        h = Herbie(
            pd.to_datetime(date).strftime("%Y-%m-%d 12:00"),
            model="hrrr",
            product="sfc",
            fxx=0
        )

        for feature_name, search_string in hrrr_variable_map.items():
            grib_file = None

            try:
                grib_file = h.download(searchString=search_string)

                value = extract_point_from_grib(
                    grib_file=grib_file,
                    site_lat=site_lat,
                    site_lon=site_lon
                )

                row[feature_name] = value

            except Exception as e:
                print("Failed variable:", feature_name, "for", date, e)
                row[feature_name] = np.nan

            finally:
                if grib_file is not None and os.path.exists(grib_file):
                    os.remove(grib_file)

                gc.collect()

    except Exception as e:
        print("Failed date:", date, e)
        row["hrrr_error"] = str(e)

    return row

In [ ]:
# Pull sampled HRRR weather values
# This uses the sampled dates from Cell 24.
# Start with sample_step = 15 in Cell 24.
# If this works, you can later try sample_step = 10 or 7.

hrrr_extracted_path = "/content/drive/MyDrive/ResearchPG/Datasets/HRRR_extracted_sampled.csv"

hrrr_rows = []

for date in tqdm(dates):
    row = get_hrrr_features_for_date(date, site_lat, site_lon)
    hrrr_rows.append(row)

    # Save after every date so progress is not lost
    pd.DataFrame(hrrr_rows).to_csv(hrrr_extracted_path, index=False)

    print("Saved HRRR values for:", pd.to_datetime(date).strftime("%Y-%m-%d"))

    gc.collect()

hrrr_sampled = pd.DataFrame(hrrr_rows)

print("Completed sampled HRRR extraction")
print("Rows:", len(hrrr_sampled))
print("Columns:", hrrr_sampled.columns.tolist())

hrrr_sampled.head()

In [ ]:
# Clean HRRR features and create useful weather variables

hrrr_daily_clean = hrrr_sampled.copy()

hrrr_daily_clean["Date"] = pd.to_datetime(hrrr_daily_clean["Date"])

if "hrrr_error" in hrrr_daily_clean.columns:
    hrrr_daily_clean = hrrr_daily_clean.drop(columns=["hrrr_error"])

# Convert Kelvin to Celsius
if "temp_2m_K" in hrrr_daily_clean.columns:
    hrrr_daily_clean["temp_2m_C"] = hrrr_daily_clean["temp_2m_K"] - 273.15

if "dewpoint_2m_K" in hrrr_daily_clean.columns:
    hrrr_daily_clean["dewpoint_2m_C"] = hrrr_daily_clean["dewpoint_2m_K"] - 273.15

# Calculate wind speed from u and v wind components
if "u_wind_10m" in hrrr_daily_clean.columns and "v_wind_10m" in hrrr_daily_clean.columns:
    hrrr_daily_clean["wind_speed_10m"] = np.sqrt(
        hrrr_daily_clean["u_wind_10m"] ** 2 + hrrr_daily_clean["v_wind_10m"] ** 2
    )

print("Cleaned HRRR columns:")
print(hrrr_daily_clean.columns.tolist())

hrrr_daily_clean.head()

In [ ]:
#Convert sampled HRRR data back into daily data
# Your main PM2.5 dataset is already daily.
# This fills HRRR values for the missing days between sampled dates.

full_date_range = pd.DataFrame({
    "Date": pd.date_range(
        start=merged_pm25_ndvi_fire["Date"].min(),
        end=merged_pm25_ndvi_fire["Date"].max(),
        freq="D"
    )
})

hrrr_daily_filled = full_date_range.merge(
    hrrr_daily_clean,
    on="Date",
    how="left"
)

numeric_cols = hrrr_daily_filled.select_dtypes(include="number").columns

hrrr_daily_filled[numeric_cols] = hrrr_daily_filled[numeric_cols].interpolate(
    method="linear"
)

hrrr_daily_filled[numeric_cols] = hrrr_daily_filled[numeric_cols].bfill().ffill()

print("Daily HRRR rows:", len(hrrr_daily_filled))
print("Missing values after filling:")
print(hrrr_daily_filled.isna().sum())

hrrr_daily_filled.head()

In [ ]:
# Save daily HRRR dataset

hrrr_daily_path = "/content/drive/MyDrive/ResearchPG/Datasets/HRRR_daily_filled.csv"

hrrr_daily_filled.to_csv(hrrr_daily_path, index=False)

print("Saved daily HRRR dataset to:", hrrr_daily_path)

In [ ]:
#Merge HRRR with PM2.5 + NDVI + wildfire data

merged_pm25_ndvi_fire["Date"] = pd.to_datetime(merged_pm25_ndvi_fire["Date"])
hrrr_daily_filled["Date"] = pd.to_datetime(hrrr_daily_filled["Date"])

final_merged_dataset = merged_pm25_ndvi_fire.merge(
    hrrr_daily_filled,
    on="Date",
    how="left"
)

print("Final dataset shape:", final_merged_dataset.shape)
print("Final dataset columns:")
print(final_merged_dataset.columns.tolist())

final_merged_dataset.head()

In [ ]:
# Save final PM2.5 + NDVI + wildfire + HRRR dataset

final_merged_path = "/content/drive/MyDrive/ResearchPG/Datasets/PM25_NDVI_Wildfire_HRRR_merged.csv"

final_merged_dataset.to_csv(final_merged_path, index=False)

print("Saved final merged dataset to:", final_merged_path)

In [ ]:
#Prepare final dataset for machine learning

import pandas as pd
import numpy as np

model_df = final_merged_dataset.copy()

# Make sure Date is datetime
model_df["Date"] = pd.to_datetime(model_df["Date"])

# Sort by date because this is time-based data
model_df = model_df.sort_values("Date").reset_index(drop=True)

print("Dataset shape:", model_df.shape)
print("Columns:")
print(model_df.columns.tolist())

model_df.head()

In [ ]:
#Identify the PM2.5 target column

possible_targets = [
    "Daily Mean PM2.5 Concentration",
    "PM2.5",
    "pm25",
    "PM25",
    "pm2.5"
]

target_col = None

for col in possible_targets:
    if col in model_df.columns:
        target_col = col
        break

if target_col is None:
    print("Could not automatically find PM2.5 column.")
    print("Available columns:")
    print(model_df.columns.tolist())
else:
    print("Target column:", target_col)

In [ ]:
# Create date-based features
# These help the model learn seasonal patterns.

model_df["month"] = model_df["Date"].dt.month
model_df["day_of_year"] = model_df["Date"].dt.dayofyear
model_df["day_of_week"] = model_df["Date"].dt.dayofweek

model_df[["Date", "month", "day_of_year", "day_of_week"]].head()

In [ ]:
#Create lag features
# These use previous PM2.5 values to predict future PM2.5.

model_df["pm25_lag_1"] = model_df[target_col].shift(1)
model_df["pm25_lag_3"] = model_df[target_col].shift(3)
model_df["pm25_lag_7"] = model_df[target_col].shift(7)

model_df["pm25_rolling_3"] = model_df[target_col].rolling(window=3).mean()
model_df["pm25_rolling_7"] = model_df[target_col].rolling(window=7).mean()

model_df.head(10)

In [ ]:
#Select numeric features for modeling

exclude_cols = ["Date", target_col]

numeric_cols = model_df.select_dtypes(include=["number"]).columns.tolist()

feature_cols = [col for col in numeric_cols if col not in exclude_cols]

print("Number of features:", len(feature_cols))
print("Features used:")
print(feature_cols)

In [ ]:
# Clean dataset for modeling

ml_df = model_df[["Date", target_col] + feature_cols].copy()

# Remove rows with missing target
ml_df = ml_df.dropna(subset=[target_col])

# Fill missing feature values
ml_df[feature_cols] = ml_df[feature_cols].interpolate(method="linear")
ml_df[feature_cols] = ml_df[feature_cols].bfill().ffill()

# Drop remaining missing rows caused by lag features
ml_df = ml_df.dropna().reset_index(drop=True)

print("ML dataset shape:", ml_df.shape)
print("Missing values:")
print(ml_df.isna().sum())

ml_df.head()

In [ ]:
# Time-based train/test split
# We use the earlier 80% of dates for training and the later 20% for testing.

split_index = int(len(ml_df) * 0.8)

train_df = ml_df.iloc[:split_index]
test_df = ml_df.iloc[split_index:]

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Train date range:", train_df["Date"].min(), "to", train_df["Date"].max())
print("Test date range:", test_df["Date"].min(), "to", test_df["Date"].max())

In [ ]:
# Scaled model setup
# This cell rescales the model variables before running any models.
# Features are scaled with MinMaxScaler.
# The target is also scaled for model training, then predictions are converted back to PM2.5 units for evaluation.

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

# Scale X features between 0 and 1
x_scaler = MinMaxScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

# Scale y target between 0 and 1
y_scaler = MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(np.array(y_train).reshape(-1, 1)).ravel()
y_test_scaled = y_scaler.transform(np.array(y_test).reshape(-1, 1)).ravel()

print("X_train_scaled min:", X_train_scaled.min())
print("X_train_scaled max:", X_train_scaled.max())
print("y_train_scaled min:", y_train_scaled.min())
print("y_train_scaled max:", y_train_scaled.max())

In [ ]:
# Scaled baseline model
# Baseline uses the previous day's PM2.5 value as the prediction.
# The lag values are scaled first, then converted back to PM2.5 units for evaluation.

baseline_lag_raw = np.array(test_df["pm25_lag_1"]).reshape(-1, 1)
baseline_lag_scaled = y_scaler.transform(baseline_lag_raw).ravel()

baseline_preds = y_scaler.inverse_transform(
    baseline_lag_scaled.reshape(-1, 1)
).ravel()

baseline_mae = mean_absolute_error(y_test, baseline_preds)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_preds))
baseline_r2 = r2_score(y_test, baseline_preds)
print("Scaled Baseline Model Results")
print("MAE:", baseline_mae)
print("RMSE:", baseline_rmse)
print("R²:", baseline_r2)

In [ ]:
# Scaled Linear Regression model

from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train_scaled)

linear_preds_scaled = linear_model.predict(X_test_scaled)

linear_preds = y_scaler.inverse_transform(
    linear_preds_scaled.reshape(-1, 1)
).ravel()

linear_mae = mean_absolute_error(y_test, linear_preds)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_preds))
linear_r2 = r2_score(y_test, linear_preds)

print("Scaled Linear Regression Results")
print("MAE:", linear_mae)
print("RMSE:", linear_rmse)
print("R²:", linear_r2)

In [ ]:
# Random Forest tuned with GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

rf_param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [4, 6, 8, 12, None],
    "min_samples_leaf": [1, 2, 5]
}

tscv = TimeSeriesSplit(n_splits=5)

rf_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

rf_search.fit(X_train_scaled, y_train_scaled)

print("Best RF params:", rf_search.best_params_)

rf_model = rf_search.best_estimator_
rf_preds_scaled = rf_model.predict(X_test_scaled)
rf_preds = y_scaler.inverse_transform(rf_preds_scaled.reshape(-1, 1)).ravel()

rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2 = r2_score(y_test, rf_preds)

print("Tuned Random Forest Results")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R²:", rf_r2)

In [ ]:
# Gradient Boosting tuned with GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

gb_param_grid = {
    "n_estimators": [200, 300, 400],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4]
}

tscv = TimeSeriesSplit(n_splits=5)

gb_search = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    gb_param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

gb_search.fit(X_train_scaled, y_train_scaled)

print("Best GB params:", gb_search.best_params_)

gb_model = gb_search.best_estimator_
gb_preds_scaled = gb_model.predict(X_test_scaled)
gb_preds = y_scaler.inverse_transform(gb_preds_scaled.reshape(-1, 1)).ravel()

gb_mae = mean_absolute_error(y_test, gb_preds)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_preds))
gb_r2 = r2_score(y_test, gb_preds)

print("Tuned Gradient Boosting Results")
print("MAE:", gb_mae)
print("RMSE:", gb_rmse)
print("R²:", gb_r2)

In [ ]:
# Polynomial Regression with Ridge and Lasso — full grid over degree AND alpha
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

tscv = TimeSeriesSplit(n_splits=5)
results = []

for degree in range(1, 5):

    # Ridge: search alpha at this degree
    ridge_pipe = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("regressor", Ridge())
    ])
    ridge_grid = GridSearchCV(
        ridge_pipe,
        {"regressor__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
        cv=tscv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )
    ridge_grid.fit(X_train_scaled, y_train)
    ridge_preds = ridge_grid.predict(X_test_scaled)

    results.append({
        "Penalty": "Ridge",
        "Degree": degree,
        "Best Alpha": ridge_grid.best_params_["regressor__alpha"],
        "MAE": mean_absolute_error(y_test, ridge_preds),
        "MSE": mean_squared_error(y_test, ridge_preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, ridge_preds)),
        "R²": r2_score(y_test, ridge_preds)
    })

    # Lasso: search alpha at this degree
    lasso_pipe = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("regressor", Lasso(max_iter=10000))
    ])
    lasso_grid = GridSearchCV(
        lasso_pipe,
        {"regressor__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0]},
        cv=tscv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )
    lasso_grid.fit(X_train_scaled, y_train)
    lasso_preds = lasso_grid.predict(X_test_scaled)

    results.append({
        "Penalty": "Lasso",
        "Degree": degree,
        "Best Alpha": lasso_grid.best_params_["regressor__alpha"],
        "MAE": mean_absolute_error(y_test, lasso_preds),
        "MSE": mean_squared_error(y_test, lasso_preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, lasso_preds)),
        "R²": r2_score(y_test, lasso_preds)
    })

poly_penalty_results_df = pd.DataFrame(results).sort_values("RMSE")
print(poly_penalty_results_df)

In [ ]:
# MLP tuned with GridSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

mlp_param_grid = {
    "hidden_layer_sizes": [(64, 32), (128, 64, 32), (128, 64, 32, 16)],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate_init": [0.001, 0.01]
}

tscv = TimeSeriesSplit(n_splits=5)

mlp_search = GridSearchCV(
    MLPRegressor(activation="relu", solver="adam", max_iter=1000, random_state=42),
    mlp_param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

mlp_search.fit(X_train_scaled, y_train_scaled)

print("Best MLP params:", mlp_search.best_params_)

mlp_model = mlp_search.best_estimator_
mlp_preds_scaled = mlp_model.predict(X_test_scaled)
mlp_preds = y_scaler.inverse_transform(mlp_preds_scaled.reshape(-1, 1)).ravel()

mlp_mae = mean_absolute_error(y_test, mlp_preds)
mlp_rmse = np.sqrt(mean_squared_error(y_test, mlp_preds))
mlp_r2 = r2_score(y_test, mlp_preds)

print("Tuned MLP Results")
print("MAE:", mlp_mae)
print("RMSE:", mlp_rmse)
print("R²:", mlp_r2)

In [ ]:
# Scaled MLP feature extraction + Linear Regression
# This uses the hidden-layer representation from the scaled MLP as extra features for Linear Regression.

def get_mlp_features(model, X):
    layer_output = X

    # Pass through all hidden layers, excluding the final output layer
    for i in range(len(model.coefs_) - 1):
        layer_output = np.dot(layer_output, model.coefs_[i]) + model.intercepts_[i]
        layer_output = np.maximum(layer_output, 0)  # ReLU

    return layer_output

mlp_train_features = get_mlp_features(mlp_model, X_train_scaled)
mlp_test_features = get_mlp_features(mlp_model, X_test_scaled)

X_train_mlp_lr = np.hstack([X_train_scaled, mlp_train_features])
X_test_mlp_lr = np.hstack([X_test_scaled, mlp_test_features])

mlp_lr_model = LinearRegression()
mlp_lr_model.fit(X_train_mlp_lr, y_train_scaled)

mlp_lr_preds_scaled = mlp_lr_model.predict(X_test_mlp_lr)

mlp_lr_preds = y_scaler.inverse_transform(
    mlp_lr_preds_scaled.reshape(-1, 1)
).ravel()

mlp_lr_mae = mean_absolute_error(y_test, mlp_lr_preds)
mlp_lr_rmse = np.sqrt(mean_squared_error(y_test, mlp_lr_preds))
mlp_lr_r2 = r2_score(y_test, mlp_lr_preds)

print("Scaled MLP + Linear Regression Results")
print("MAE:", mlp_lr_mae)
print("RMSE:", mlp_lr_rmse)
print("R²:", mlp_lr_r2)

In [ ]:
# Scaled ARIMA model
# ARIMA uses only the scaled target series.
# Predictions are converted back to PM2.5 units before evaluation.

from statsmodels.tsa.arima.model import ARIMA

arima_model = ARIMA(
    y_train_scaled,
    order=(3, 1, 2)
)

arima_fit = arima_model.fit()

arima_preds_scaled = arima_fit.forecast(steps=len(y_test_scaled))

arima_preds = y_scaler.inverse_transform(
    np.array(arima_preds_scaled).reshape(-1, 1)
).ravel()

arima_mae = mean_absolute_error(y_test, arima_preds)
arima_rmse = np.sqrt(mean_squared_error(y_test, arima_preds))
arima_r2 = r2_score(y_test, arima_preds)

print("Scaled ARIMA Results")
print("MAE:", arima_mae)
print("RMSE:", arima_rmse)
print("R²:", arima_r2)

In [ ]:
# Scaled SARIMAX model
# SARIMAX uses the scaled target series and scaled external variables.

from statsmodels.tsa.statespace.sarimax import SARIMAX

sarimax_model = SARIMAX(
    y_train_scaled,
    exog=X_train_scaled,
    order=(1, 1, 1),
    seasonal_order=(0, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarimax_fit = sarimax_model.fit(maxiter=200, disp=False)

sarimax_preds_scaled = sarimax_fit.predict(
    start=len(y_train_scaled),
    end=len(y_train_scaled) + len(y_test_scaled) - 1,
    exog=X_test_scaled
)

sarimax_preds = y_scaler.inverse_transform(
    np.array(sarimax_preds_scaled).reshape(-1, 1)
).ravel()

sarimax_mae = mean_absolute_error(y_test, sarimax_preds)
sarimax_rmse = np.sqrt(mean_squared_error(y_test, sarimax_preds))
sarimax_r2 = r2_score(y_test, sarimax_preds)

print("Scaled SARIMAX Results")
print("MAE:", sarimax_mae)
print("RMSE:", sarimax_rmse)
print("R²:", sarimax_r2)

In [ ]:
# Polynomial Regression with Ridge and Lasso penalties

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

results = []

# Test polynomial degrees
for degree in range(1, 5):


    poly_ridge_model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("regressor", Ridge(alpha=1.0))
    ])

    poly_ridge_model.fit(X_train_scaled, y_train)

    ridge_preds = poly_ridge_model.predict(X_test_scaled)
    ridge_mae = mean_absolute_error(y_test, ridge_preds)
    ridge_mse = mean_squared_error(y_test, ridge_preds)
    ridge_rmse = np.sqrt(ridge_mse)
    ridge_r2 = r2_score(y_test, ridge_preds)

    results.append({
        "Penalty": "Ridge",
        "Degree": degree,
        "MAE": ridge_mae,
        "MSE": ridge_mse,
        "RMSE": ridge_rmse,
        "R²": ridge_r2
    })


    poly_lasso_model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("regressor", Lasso(alpha=0.001, max_iter=10000))
    ])

    poly_lasso_model.fit(X_train_scaled, y_train)

    lasso_preds = poly_lasso_model.predict(X_test_scaled)
    lasso_mae = mean_absolute_error(y_test, lasso_preds)
    lasso_mse = mean_squared_error(y_test, lasso_preds)
    lasso_rmse = np.sqrt(lasso_mse)
    lasso_r2 = r2_score(y_test, lasso_preds)

    results.append({
        "Penalty": "Lasso",
        "MAE": lasso_mae,
        "Degree": degree,
        "MSE": lasso_mse,
        "RMSE": lasso_rmse,
        "R²": lasso_r2
    })

# Create results table
poly_fixed_alpha_results_df = pd.DataFrame(results)

# Sort by RMSE
poly_fixed_alpha_results_df = poly_fixed_alpha_results_df.sort_values("RMSE")

print(poly_fixed_alpha_results_df)

In [ ]:
# Final comparison table for all scaled models
scaled_results_df = pd.DataFrame({
    "Model": ["Baseline", "Linear Regression", "Random Forest", "Gradient Boosting", "MLP", "MLP + Linear Regression", "ARIMA", "SARIMAX"],
    "MAE": [baseline_mae, linear_mae, rf_mae, gb_mae, mlp_mae, mlp_lr_mae, arima_mae, sarimax_mae],
    "RMSE": [baseline_rmse, linear_rmse, rf_rmse, gb_rmse, mlp_rmse, mlp_lr_rmse, arima_rmse, sarimax_rmse],
    "R²": [baseline_r2, linear_r2, rf_r2, gb_r2, mlp_r2, mlp_lr_r2, arima_r2, sarimax_r2]
})

scaled_results_df = scaled_results_df.sort_values("R²", ascending=False)
scaled_results_df

In [ ]:
# Save scaled model predictions

scaled_predictions_df = test_df[["Date", target_col]].copy()

scaled_predictions_df["Baseline_Prediction"] = baseline_preds
scaled_predictions_df["Linear_Regression_Prediction"] = linear_preds
scaled_predictions_df["Random_Forest_Prediction"] = rf_preds
scaled_predictions_df["Gradient_Boosting_Prediction"] = gb_preds
scaled_predictions_df["MLP_Prediction"] = mlp_preds
scaled_predictions_df["MLP_Linear_Regression_Prediction"] = mlp_lr_preds
scaled_predictions_df["ARIMA_Prediction"] = arima_preds
scaled_predictions_df["SARIMAX_Prediction"] = sarimax_preds

scaled_predictions_path = "/content/drive/MyDrive/ResearchPG/Datasets/PM25_scaled_model_predictions.csv"

scaled_predictions_df.to_csv(scaled_predictions_path, index=False)

print("Saved scaled model predictions to:", scaled_predictions_path)

scaled_predictions_df.head()

In [ ]:
# Compare best polynomial penalty model to all other models

import pandas as pd

best_poly_penalty = poly_penalty_results_df.sort_values("RMSE").iloc[0]

best_poly_model_name = (
    "Polynomial Regression + "
    + best_poly_penalty["Penalty"]
    + " Penalty (Degree "
    + str(int(best_poly_penalty["Degree"]))
    + ")"
)

final_comparison_df = pd.DataFrame({
    "Model": [
        "Baseline",
        "Linear Regression",
        best_poly_model_name,
        "Random Forest",
        "Gradient Boosting",
        "MLP",
        "MLP + Linear Regression",
        "ARIMA",
        "SARIMAX"
    ],

    "MAE": [
        baseline_mae,
        linear_mae,
        np.nan,
        rf_mae,
        gb_mae,
        mlp_mae,
        mlp_lr_mae,
        arima_mae,
        sarimax_mae
    ],

    "MSE": [
        baseline_rmse**2,
        linear_rmse**2,
        best_poly_penalty["MSE"],
        rf_rmse**2,
        gb_rmse**2,
        mlp_rmse**2,
        mlp_lr_rmse**2,
        arima_rmse**2,
        sarimax_rmse**2
    ],

    "RMSE": [
        baseline_rmse,
        linear_rmse,
        best_poly_penalty["RMSE"],
        rf_rmse,
        gb_rmse,
        mlp_rmse,
        mlp_lr_rmse,
        arima_rmse,
        sarimax_rmse
    ],

    "R²": [
        baseline_r2,
        linear_r2,
        best_poly_penalty["R²"],
        rf_r2,
        gb_r2,
        mlp_r2,
        mlp_lr_r2,
        arima_r2,
        sarimax_r2
    ]
})

final_comparison_df = final_comparison_df.sort_values(
    "R²",
    ascending=False
)

print(final_comparison_df)

In [ ]:
#Final comparison table:

if "MAE" in poly_penalty_results_df.columns:
  final_comparison_df.loc[
      final_comparison_df["Model"].str.contains("Polynomial"),
      "MAE"
  ] = best_poly_penalty["MAE"]

#Sort by best R² value:
final_comparison_df = final_comparison_df.sort_values("R²", ascending=False)

#Display final table:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

final_comparison_df

In [ ]:
#Print out final results of the Polynomial Regression model with Ridge + Lasso to see comparisons
poly_penalty_results_df.sort_values("RMSE")

In [ ]:
# CNN tuned with Keras Tuner (RandomSearch)
!pip install -q keras-tuner
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train_cnn = np.array(X_train_scaled).reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = np.array(X_test_scaled).reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

def build_cnn(hp):
    model = Sequential()
    model.add(Conv1D(
        filters=hp.Choice("filters_1", [16, 32, 64]),
        kernel_size=3, activation="relu",
        input_shape=(X_train_cnn.shape[1], 1)
    ))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Conv1D(filters=hp.Choice("filters_2", [32, 64, 128]), kernel_size=3, activation="relu"))
    model.add(Dropout(hp.Choice("dropout_1", [0.1, 0.2, 0.3])))
    model.add(Flatten())
    model.add(Dense(hp.Choice("dense_units", [32, 64, 128]), activation="relu"))
    model.add(Dropout(hp.Choice("dropout_2", [0.1, 0.2, 0.3])))
    model.add(Dense(1))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(hp.Choice("lr", [0.01, 0.001, 0.0005])),
        loss="mse"
    )
    return model

tuner = kt.RandomSearch(
    build_cnn,
    objective="val_loss",
    max_trials=15,
    overwrite=True,
    directory="cnn_tuning",
    project_name="pm25_cnn"
)

early_stop = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

tuner.search(
    X_train_cnn, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    callbacks=[early_stop],
    verbose=0
)

cnn_model = tuner.get_best_models(num_models=1)[0]
print("Best CNN hyperparameters:", tuner.get_best_hyperparameters(1)[0].values)

cnn_preds = cnn_model.predict(X_test_cnn).flatten()

cnn_mae = mean_absolute_error(y_test, cnn_preds)
cnn_mse = mean_squared_error(y_test, cnn_preds)
cnn_rmse = np.sqrt(cnn_mse)
cnn_r2 = r2_score(y_test, cnn_preds)

print("Tuned CNN Results:")
print("MAE:", cnn_mae)
print("MSE:", cnn_mse)
print("RMSE:", cnn_rmse)
print("R²:", cnn_r2)

In [ ]:
# Zero-shot rule-based model
# This model is not trained on the dataset.
# It uses fixed assumptions to predict PM2.5 from environmental features.

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

zero_shot_df = test_df.copy()

# Start with persistence from the previous day
zero_shot_preds = zero_shot_df["pm25_lag_1"].copy()

# Add wildfire effect if available
if "fire_count" in zero_shot_df.columns:
    zero_shot_preds += 0.15 * zero_shot_df["fire_count"]

if "mean_frp" in zero_shot_df.columns:
    zero_shot_preds += 0.05 * zero_shot_df["mean_frp"]

if "max_frp" in zero_shot_df.columns:
    zero_shot_preds += 0.02 * zero_shot_df["max_frp"]

# Add wind effect if available
if "wind_speed_10m" in zero_shot_df.columns:
    zero_shot_preds += 0.10 * zero_shot_df["wind_speed_10m"]

# Add temperature effect if available
if "temp_2m_C" in zero_shot_df.columns:
    zero_shot_preds += 0.03 * zero_shot_df["temp_2m_C"]

# Reduce PM2.5 if NDVI is higher
if "NDVI" in zero_shot_df.columns:
    zero_shot_preds -= 2.0 * zero_shot_df["NDVI"]

if "mean_ndvi" in zero_shot_df.columns:
    zero_shot_preds -= 2.0 * zero_shot_df["mean_ndvi"]

# Prevent impossible negative PM2.5 predictions
zero_shot_preds = np.maximum(zero_shot_preds, 0)

# Evaluate
zero_shot_mae = mean_absolute_error(y_test, zero_shot_preds)
zero_shot_mse = mean_squared_error(y_test, zero_shot_preds)
zero_shot_rmse = np.sqrt(zero_shot_mse)
zero_shot_r2 = r2_score(y_test, zero_shot_preds)

print("Zero-Shot Rule-Based Model Results")
print("MAE:", zero_shot_mae)
print("MSE:", zero_shot_mse)
print("RMSE:", zero_shot_rmse)
print("R²:", zero_shot_r2)

In [ ]:
zero_shot_row = pd.DataFrame({
    "Model": ["Zero-Shot Rule-Based Model"],
    "MAE": [zero_shot_mae],
    "MSE": [zero_shot_mse],
    "RMSE": [zero_shot_rmse],
    "R²": [zero_shot_r2]
})

final_comparison_with_zero_shot = pd.concat(
    [final_comparison_df, zero_shot_row],
    ignore_index=True
)

final_comparison_with_zero_shot = final_comparison_with_zero_shot.sort_values("R²", ascending=False)

final_comparison_with_zero_shot

In [ ]:
# Elastic Net Regression Model
# Combines Ridge and Lasso penalties

from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

elastic_model = ElasticNet(
    alpha=0.001,
    l1_ratio=0.5,
    max_iter=10000,
    random_state=42
)

elastic_model.fit(X_train_scaled, y_train)

elastic_preds = elastic_model.predict(X_test_scaled)

elastic_mae = mean_absolute_error(y_test, elastic_preds)
elastic_mse = mean_squared_error(y_test, elastic_preds)
elastic_rmse = np.sqrt(elastic_mse)
elastic_r2 = r2_score(y_test, elastic_preds)

print("Elastic Net Results")
print("MAE:", elastic_mae)
print("MSE:", elastic_mse)
print("RMSE:", elastic_rmse)
print("R²:", elastic_r2)

In [ ]:
elastic_row = pd.DataFrame({
    "Model": ["Elastic Net"],
    "MAE": [elastic_mae],
    "MSE": [elastic_mse],
    "RMSE": [elastic_rmse],
    "R²": [elastic_r2]
})

final_comparison_with_elastic = pd.concat(
    [final_comparison_df, elastic_row],
    ignore_index=True
)

final_comparison_with_elastic = final_comparison_with_elastic.sort_values("R²", ascending=False)

final_comparison_with_elastic

In [ ]:
#LLM Few-Shot Prompting Model:
#This gives the LLM a few examples, then asks it to predict PM2.5 for new rows.

from transformers import pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
import re

In [ ]:
#Load Text Generation LLM
llm = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens = 20
)

In [ ]:
#Pick a few examples for the prompt:

few_shot_examples = train_df.sample(
    n=5,
    random_state = 42
)

In [ ]:
#Helper function to extract a number from LLM output:

def extract_number(text):
  match = re.search(r"[-+]?\d*\.\d+|\d+", text)
  if match:
    return float(match.group())

  return np.nan

In [ ]:
# Build few-shot prompt

def build_few_shot_prompt(test_row):
    prompt = """
You are predicting daily mean PM2.5 concentration in micrograms per cubic meter.
Use the examples below to learn the pattern.
Return only one number.

Examples:
"""

    for _, row in few_shot_examples.iterrows():
        prompt += f"""
Input:
Previous day PM2.5: {row.get("pm25_lag_1", "unknown")}
3-day PM2.5 lag: {row.get("pm25_lag_3", "unknown")}
7-day PM2.5 lag: {row.get("pm25_lag_7", "unknown")}
NDVI: {row.get("mean_ndvi", row.get("NDVI", "unknown"))}
Fire count: {row.get("fire_count", "unknown")}
Mean FRP: {row.get("mean_frp", row.get("frp_mean", "unknown"))}
Max FRP: {row.get("max_frp", row.get("frp_max", "unknown"))}
Temperature: {row.get("temp_2m_C", "unknown")}
Wind speed: {row.get("wind_speed_10m", row.get("wind_speed", "unknown"))}

Output:
{row[target_col]}
"""

    prompt += f"""
Now predict this case:

Input:
Previous day PM2.5: {test_row.get("pm25_lag_1", "unknown")}
3-day PM2.5 lag: {test_row.get("pm25_lag_3", "unknown")}
7-day PM2.5 lag: {test_row.get("pm25_lag_7", "unknown")}
NDVI: {test_row.get("mean_ndvi", test_row.get("NDVI", "unknown"))}
Fire count: {test_row.get("fire_count", "unknown")}
Mean FRP: {test_row.get("mean_frp", test_row.get("frp_mean", "unknown"))}
Max FRP: {test_row.get("max_frp", test_row.get("frp_max", "unknown"))}
Temperature: {test_row.get("temp_2m_C", "unknown")}
Wind speed: {test_row.get("wind_speed_10m", test_row.get("wind_speed", "unknown"))}

Output:
"""

    return prompt

In [ ]:
# Run LLM few-shot predictions
# Start small first because LLM inference is slow.

llm_few_shot_preds = []

for i, row in test_df.iterrows():
    prompt = build_few_shot_prompt(row)

    output = llm(prompt)[0]["generated_text"]

    # Only parse the part after the final Output:
    final_answer = output.split("Output:")[-1]

    pred = extract_number(final_answer)

    # fallback if model fails to output a number
    if np.isnan(pred):
        pred = row["pm25_lag_1"]

    llm_few_shot_preds.append(pred)

llm_few_shot_preds = np.array(llm_few_shot_preds)

print("Finished LLM few-shot predictions")

In [ ]:
# Evaluate LLM few-shot model

llm_few_shot_mae = mean_absolute_error(y_test, llm_few_shot_preds)
llm_few_shot_mse = mean_squared_error(y_test, llm_few_shot_preds)
llm_few_shot_rmse = np.sqrt(llm_few_shot_mse)
llm_few_shot_r2 = r2_score(y_test, llm_few_shot_preds)

print("LLM Few-Shot Model Results")
print("MAE:", llm_few_shot_mae)
print("MSE:", llm_few_shot_mse)
print("RMSE:", llm_few_shot_rmse)
print("R²:", llm_few_shot_r2)

In [ ]:
# Add LLM few-shot model to comparison table

llm_few_shot_row = pd.DataFrame({
    "Model": ["LLM Few-Shot Prompting"],
    "MAE": [llm_few_shot_mae],
    "MSE": [llm_few_shot_mse],
    "RMSE": [llm_few_shot_rmse],
    "R²": [llm_few_shot_r2]
})

final_comparison_with_llm = pd.concat(
    [final_comparison_df, llm_few_shot_row],
    ignore_index=True
)

final_comparison_with_llm = final_comparison_with_llm.sort_values("R²", ascending=False)

final_comparison_with_llm

In [ ]:
pd.DataFrame({"Neural Network Type": ["MLP", "CNN", "MLP + Linear Regression (Hybrid)", "LLM Few-Shot (Transformer, distilgpt2)"], "R²": [mlp_r2, cnn_r2, mlp_lr_r2, llm_few_shot_r2], "MAE": [mlp_mae, cnn_mae, mlp_lr_mae, llm_few_shot_mae], "RMSE": [mlp_rmse, cnn_rmse, mlp_lr_rmse, llm_few_shot_rmse]}).sort_values("R²", ascending=False)

In [ ]:
# Grand comparison table: every model, tuned results included

# Safety check: make sure poly_penalty_results_df is the grid-search version
# (the one from the "full grid over degree AND alpha" cell), not the
# fixed-alpha version from the other polynomial cell. If cell 52 (fixed alpha)
# ran after cell 47 (grid search), it will have overwritten this dataframe
# and this check will catch it before causing a confusing error below.
if "Best Alpha" not in poly_penalty_results_df.columns:
    raise RuntimeError(
        "poly_penalty_results_df is missing the 'Best Alpha' column. "
        "This means the fixed-alpha polynomial cell ran after the grid-search "
        "polynomial cell and overwrote it. Re-run the 'full grid over degree "
        "AND alpha' cell before running this one."
    )

best_poly = poly_penalty_results_df.sort_values("RMSE").iloc[0]
best_poly_name = f"Polynomial Regression + {best_poly['Penalty']} (Degree {int(best_poly['Degree'])}, alpha={best_poly['Best Alpha']})"

all_models_comparison_df = pd.DataFrame({
    "Model": [
        "Baseline",
        "Linear Regression",
        "Random Forest (tuned)",
        "Gradient Boosting (tuned)",
        best_poly_name,
        "MLP (tuned)",
        "MLP + Linear Regression",
        "ARIMA",
        "SARIMAX",
        "CNN (tuned)",
        "Zero-Shot Rule-Based",
        "Elastic Net",
        "LLM Few-Shot Prompting"
    ],
    "MAE": [
        baseline_mae, linear_mae, rf_mae, gb_mae, best_poly["MAE"],
        mlp_mae, mlp_lr_mae, arima_mae, sarimax_mae, cnn_mae,
        zero_shot_mae, elastic_mae, llm_few_shot_mae
    ],
    "MSE": [
        baseline_rmse**2, linear_rmse**2, rf_rmse**2, gb_rmse**2, best_poly["MSE"],
        mlp_rmse**2, mlp_lr_rmse**2, arima_rmse**2, sarimax_rmse**2, cnn_mse,
        zero_shot_mse, elastic_mse, llm_few_shot_mse
    ],
    "RMSE": [
        baseline_rmse, linear_rmse, rf_rmse, gb_rmse, best_poly["RMSE"],
        mlp_rmse, mlp_lr_rmse, arima_rmse, sarimax_rmse, cnn_rmse,
        zero_shot_rmse, elastic_rmse, llm_few_shot_rmse
    ],
    "R²": [
        baseline_r2, linear_r2, rf_r2, gb_r2, best_poly["R²"],
        mlp_r2, mlp_lr_r2, arima_r2, sarimax_r2, cnn_r2,
        zero_shot_r2, elastic_r2, llm_few_shot_r2
    ]
})

all_models_comparison_df = all_models_comparison_df.sort_values("R²", ascending=False).reset_index(drop=True)
all_models_comparison_df

In [ ]:
# Checking if Polynomial Regression actually beats Linear Regression and Elastic Net,
# or if the difference is just chance since the R² scores are so close.
# Using a bootstrap here since it's the same kind of test used in similar published work.

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
import numpy as np

# Refit the winning polynomial model (whatever degree/alpha the grid search picked)
best_row = poly_penalty_results_df.sort_values("RMSE").iloc[0]
best_degree = int(best_row["Degree"])
best_alpha = best_row["Best Alpha"]

winning_poly_model = Pipeline([
    ("poly", PolynomialFeatures(degree=best_degree, include_bias=False)),
    ("regressor", Lasso(alpha=best_alpha, max_iter=10000))
])
winning_poly_model.fit(X_train_scaled, y_train)
poly_preds = winning_poly_model.predict(X_test_scaled)

# Bootstrap: resample the test set with replacement a bunch of times
# and see how often each model comes out ahead
n_boot = 10000
n_test = len(y_test)
y_test_arr = np.array(y_test)

rng = np.random.default_rng(42)

rmse_diff_poly_vs_linear = []
rmse_diff_poly_vs_elastic = []

for _ in range(n_boot):
    idx = rng.integers(0, n_test, n_test)

    y_boot = y_test_arr[idx]
    poly_boot = poly_preds[idx]
    linear_boot = linear_preds[idx]
    elastic_boot = elastic_preds[idx]

    rmse_poly = np.sqrt(np.mean((y_boot - poly_boot) ** 2))
    rmse_linear = np.sqrt(np.mean((y_boot - linear_boot) ** 2))
    rmse_elastic = np.sqrt(np.mean((y_boot - elastic_boot) ** 2))

    rmse_diff_poly_vs_linear.append(rmse_linear - rmse_poly)
    rmse_diff_poly_vs_elastic.append(rmse_elastic - rmse_poly)

rmse_diff_poly_vs_linear = np.array(rmse_diff_poly_vs_linear)
rmse_diff_poly_vs_elastic = np.array(rmse_diff_poly_vs_elastic)

ci_poly_vs_linear = np.percentile(rmse_diff_poly_vs_linear, [2.5, 97.5])
ci_poly_vs_elastic = np.percentile(rmse_diff_poly_vs_elastic, [2.5, 97.5])

pct_favoring_poly_over_linear = np.mean(rmse_diff_poly_vs_linear > 0) * 100
pct_favoring_poly_over_elastic = np.mean(rmse_diff_poly_vs_elastic > 0) * 100

print("Polynomial vs Linear Regression")
print("95% CI for RMSE difference (Linear − Polynomial):", ci_poly_vs_linear)
print("Percent of bootstrap resamples favoring Polynomial:", pct_favoring_poly_over_linear)
print()
print("Polynomial vs Elastic Net")
print("95% CI for RMSE difference (Elastic − Polynomial):", ci_poly_vs_elastic)
print("Percent of bootstrap resamples favoring Polynomial:", pct_favoring_poly_over_elastic)